# Chest X-Ray Disease Classification — Kaggle Training Template

This notebook is a **template**, not a finished model. It demonstrates how to wire
any PyTorch model (ResNet50, DenseNet121, EfficientNet, ViT, ...) into this project's
shared reproducibility and experiment-tracking infrastructure (`src/utils/`).

Sections marked **TODO** are dataset/model-specific and must be filled in by whoever
owns that model. Everything else (seeding, config loading, W&B logging, checkpointing)
is shared — copy this notebook per-model rather than reinventing the tracking logic.

See `docs/experiment_policy.md` and `docs/wandb_setup.md` for the policy this notebook follows.

## 1. Environment check

In [ ]:
import sys
import platform

print('Python:', platform.python_version())
print('Executable:', sys.executable)

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA version:', torch.version.cuda)
except ImportError:
    print('PyTorch not found -- Kaggle GPU notebooks normally ship it preinstalled.')

## 2. Install dependencies

PyTorch is already provided by the Kaggle runtime -- only install the tracking/config libraries.

In [ ]:
%pip install -q wandb PyYAML

## 3. Import project code

**TODO:** attach this GitHub repository to the notebook (e.g. via 'Add Data' -> GitHub,
or `git clone` if internet access is enabled for this notebook) so `src/` is importable.
Adjust `PROJECT_ROOT` to wherever the repo ends up under `/kaggle/working` or `/kaggle/input`.

In [ ]:
import sys
from pathlib import Path

# TODO: point this at the cloned/attached repo root
PROJECT_ROOT = Path('/kaggle/working/Chest-X-ray-Disease-Detection')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    load_config,
    set_seed,
    create_generator,
    seed_worker,
    initialize_wandb,
    log_metrics,
    log_summary_metrics,
    finish_run,
    generate_run_name,
    BestCheckpointSaver,
)

## 4. W&B authentication

Use **Kaggle Secrets** (Add-ons -> Secrets) to store your W&B API key -- never paste it
directly into a cell. See `docs/wandb_setup.md` for how to create the secret.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('W&B API key loaded from Kaggle Secrets.')
except Exception as e:
    print('Could not load WANDB_API_KEY from Kaggle Secrets:', e)
    print('Set mode="disabled" below to run without authenticating.')

## 5. Load baseline YAML configuration

In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'baseline.yaml')

## 6. Display configuration

Sanity-check the config before spending GPU time on a run.

In [ ]:
import json
print(json.dumps(config, indent=2))

## 7. Set random seed

Read the seed from config -- never hardcode it at the call site.

In [ ]:
set_seed(config['experiment']['seed'])

## 8. Configure deterministic DataLoader behavior

Use this `generator` + `seed_worker` for every `DataLoader` that shuffles data.

In [ ]:
generator = create_generator(config['experiment']['seed'])

# Example (fill in once the Dataset class exists):
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config['training']['batch_size'],
#     shuffle=True,
#     worker_init_fn=seed_worker,
#     generator=generator,
#     num_workers=2,
# )

## 9. Dataset path placeholder

**TODO:** point this at the actual Kaggle dataset mount once it's attached to the notebook.

In [ ]:
# TODO: replace with the real Kaggle input path, e.g.:
# DATASET_DIR = Path('/kaggle/input/covid19-radiography-database')
DATASET_DIR = Path('/kaggle/input/REPLACE_ME')
OUTPUT_DIR = Path('/kaggle/working')

## 10. Dataset / split manifest placeholder

**Policy:** the train/val/test split must be generated once and reused by every model
-- never re-split per run. See `docs/experiment_policy.md`.

**TODO:** load the fixed split manifest (see the CSV shape documented in the policy doc)
once it has been generated for `dataset.split_version`.

In [ ]:
# TODO: load the split manifest, e.g.:
# import pandas as pd
# split_manifest = pd.read_csv(DATASET_DIR / f"splits/{config['dataset']['split_version']}.csv")
# train_df = split_manifest[split_manifest['split'] == 'train']
# val_df = split_manifest[split_manifest['split'] == 'val']
# test_df = split_manifest[split_manifest['split'] == 'test']

## 11. Model placeholder

**TODO:** replace with the real model (ResNet50, DenseNet121, EfficientNet, ViT, ...).

In [ ]:
import torch.nn as nn

# TODO: replace with the real model, e.g.:
# import timm
# model = timm.create_model(
#     config['model']['name'],
#     pretrained=config['model']['pretrained'],
#     num_classes=config['model']['num_classes'],
# )
# model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

## 12. Optimizer placeholder

In [ ]:
# TODO: replace with the real optimizer/scheduler, e.g.:
# import torch.optim as optim
# optimizer = optim.AdamW(
#     model.parameters(),
#     lr=config['training']['learning_rate'],
#     weight_decay=config['training']['weight_decay'],
# )
# criterion = nn.BCEWithLogitsLoss()  # or the appropriate loss for the task

## 13. Training loop (skeleton)

This shows the *shape* of a training epoch -- it is not a working loop until the
TODO sections above (data, model, optimizer) are filled in.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for batch in loader:  # TODO: unpack (inputs, targets) once the Dataset exists
        raise NotImplementedError('Fill in the training step for this model.')
    return running_loss / max(len(loader), 1)

## 14. Validation loop (skeleton)

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    # TODO: compute val_loss and the metrics below using the appropriate
    # multi-label/multi-class logic for this dataset -- do not fabricate values.
    # import torch
    # with torch.no_grad():
    #     for batch in loader:
    #         ...
    metrics = {
        'val_loss': None,
        'val_accuracy': None,
        'val_precision': None,
        'val_recall': None,
        'val_f1': None,
        'val_auroc': None,
    }
    return metrics

## 15. W&B run + metric logging

Start the run, then log whatever metrics are available each epoch --
`log_metrics` accepts a partial dict and drops `None` values automatically.

In [ ]:
run_name = generate_run_name(
    config['model']['name'], config['experiment']['name'], config['experiment']['seed']
)
print('Run name:', run_name)

# Use mode='disabled' while testing this template without a W&B account.
run = initialize_wandb(config, run_name=run_name, mode=None)

In [ ]:
checkpoint_saver = BestCheckpointSaver(
    run_name=run_name,
    monitor=config['checkpoint']['monitor'],
    mode=config['checkpoint']['mode'],
    checkpoint_dir=OUTPUT_DIR / 'checkpoints',
)

# TODO: uncomment once train_one_epoch/evaluate are implemented
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# for epoch in range(config['training']['epochs']):
#     train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
#     val_metrics = evaluate(model, val_loader, criterion, device)
#
#     log_metrics({
#         'epoch': epoch + 1,
#         'train_loss': train_loss,
#         'val_loss': val_metrics['val_loss'],
#         'val_accuracy': val_metrics['val_accuracy'],
#         'val_precision': val_metrics['val_precision'],
#         'val_recall': val_metrics['val_recall'],
#         'val_f1': val_metrics['val_f1'],
#         'val_auroc': val_metrics['val_auroc'],
#         'learning_rate': optimizer.param_groups[0]['lr'],
#     })
#

## 16. Best checkpoint handling

Save a checkpoint only when the monitored metric improves.

In [ ]:
#     improved = checkpoint_saver.step(
#         epoch=epoch + 1,
#         metric_value=val_metrics[config['checkpoint']['monitor']],
#         model=model,
#         optimizer=optimizer,
#         config=config,
#     )
#     if improved:
#         print(f'New best {config["checkpoint"]["monitor"]}: {checkpoint_saver.best_metric:.4f} (epoch {epoch + 1})')

## 17. Final evaluation placeholder

**TODO:** run the trained model against the held-out **test** split -- only after all
model/hyperparameter decisions were made using validation data, per the policy doc.

In [ ]:
# TODO: load checkpoint_saver.checkpoint_path and evaluate on test_df
# from src.utils import load_checkpoint
# best = load_checkpoint(checkpoint_saver.checkpoint_path)
# model.load_state_dict(best['model_state_dict'])
# test_metrics = evaluate(model, test_loader, criterion, device)

## 18. W&B run finalization

In [ ]:
log_summary_metrics(checkpoint_saver.summary())
# log_summary_metrics({'best_val_auroc': ...})  # add any other final summary fields here
finish_run()